In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV_extra_syst.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
plot_sideband = False
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, plot_sideband = plot_sideband)
mc_evt_df = mc_evt_df[mc_cumulative_masks["energy"]]

In [ ]:
#make it a slc df
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )

In [ ]:
save_result = True

# MC

In [ ]:
import hashlib
def get_MCstat_unc(evt_df, hdr_df, n_universes=100):
    # Create a unique seed based on event metadata
    # Using a hash function that's deterministic
    meta_seeds = []
    for i in tqdm(range(len(evt_df))):
        this_hdr_df = hdr_df.loc[evt_df.reset_index(level=[2]).index[i]]
        runno = this_hdr_df.run
        subrunno = this_hdr_df.subrun
        evtno = this_hdr_df.evt
        slcid = mc_evt_df.loc[mc_evt_df.index[i]].slc.self
        seed_string = f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}"
        #unique_seed = hash(f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}") % (2**32)  # Ensure it's a 32-bit integer
        unique_seed = int(
            hashlib.sha256(seed_string.encode()).hexdigest(),
            16
        ) % (2**32)
        if unique_seed in meta_seeds:
            print("duplicate seed found", unique_seed)
            break
        meta_seeds.append(unique_seed)

    # make sure the seeds are unique!
    assert len(meta_seeds) == len(set(meta_seeds))

    # generate universes
    MCstat_univ_events = np.zeros((n_universes, len(evt_df)))
    poisson_mean = 1.0

    # get Poisson weights and save to "MCstat.univ_"
    # dummy df to hold the weights -- iterative inserting causes PerformanceWarning
    mcstat_univ_cols = pd.MultiIndex.from_product(
        [["truth"], ["MCstat"], [f"univ_{i}" for i in range(n_universes)],[""],[""],[""]],
    )
    mcstat_univ_wgt = pd.DataFrame(
        1.0,
        index=evt_df.index,
        columns=mcstat_univ_cols,
    )

    for uidx in range(n_universes):
        universe_string = f"universe_{uidx}"
        universe_seed = int(
            hashlib.sha256(universe_string.encode()).hexdigest(),
            16
        ) % (2**32)
            
        poisson_weights = []
        for sidx, meta_seed in enumerate(meta_seeds):
            # Combine universe seed with event seed for unique randomness -- per event, per universe
            combined_seed = (universe_seed + meta_seed) % (2**32)
            np.random.seed(combined_seed)
            
            poisson_val = np.random.poisson(poisson_mean)
            poisson_weights.append(poisson_val)
            
        mcstat_univ_wgt[("truth","MCstat", "univ_{}".format(uidx),'','','')] = np.array(poisson_weights)
        MCstat_univ_events[uidx, :] = np.array(poisson_weights)

    evt_df = evt_df.join(mcstat_univ_wgt)
    return evt_df, MCstat_univ_events

In [ ]:
mc_evt_df, MCstat_univ_events = get_MCstat_unc(mc_evt_df, mc_hdr_df, n_universes=100)

In [ ]:

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]



syst_name = "MCstat"


save_fig = True
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/MCStat"
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)
    

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
syst_dict = {}
for var_config in var_configs:

    univ_events, cv_events = get_univ_rates(cov_type="rate", 
                                            evtdf=mc_evt_df,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=True,
                                            syst_name=syst_name)
    
    ret_MCstat = get_covariance_matrix(univ_events, cv_events)
    syst_dict[var_config.var_save_name] = ret_MCstat["cov_frac"]

    if save_fig:
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
         
        f_name = f"MCStat_syst_{var_config.var_save_name}_frac_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(ret_MCstat["cov_frac"], 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"MCStat_syst_{var_config.var_save_name}_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(ret_MCstat["cov"], 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"MCStat_syst_{var_config.var_save_name}_corr.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(ret_MCstat["corr"], 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                    save_fig=save_fig, save_name=save_full_path)


   
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/mcstat_syst_dict.npz", **syst_dict)


# Flux

In [ ]:
flux_systematics = [
    'expskin_Flux',
    'kzero_Flux',
    'horncurrent_Flux',
    'kminus_Flux',
    'kplus_Flux',
    'nucleoninexsec_Flux',
    'nucleonqexsec_Flux',
    'nucleontotxsec_Flux',
    'piminus_Flux',
    'pioninexsec_Flux',
    'pionqexsec_Flux',
    'piontotxsec_Flux',
    'piplus_Flux'
]

flux_label_map = {
    'expskin_Flux': 'Horn Skin Effect',
    'horncurrent_Flux': 'Horn Current',
    'kminus_Flux': r'$K^{-}$ Production',
    'kplus_Flux': r'$K^{+}$ Production',
    'kzero_Flux': r'$K^{0}$ Production',
    'piminus_Flux': r'$\pi^{-}$ Production',
    'piplus_Flux': r'$\pi^{+}$ Production',
    'nucleoninexsec_Flux': 'Secondary Nucleon Inelastic x-sec',
    'nucleonqexsec_Flux': 'Secondary Nucleon Quasielastic x-sec',
    'nucleontotxsec_Flux': 'Secondary Nucleon Total x-sec',
    'pioninexsec_Flux': 'Secondary Pion Inelastic x-sec',
    'pionqexsec_Flux': 'Secondary Pion Quasielastic x-sec',
    'piontotxsec_Flux': 'Secondary Pion Total x-sec'
}


In [ ]:
def corr_from_cov(cov):
    corr = np.zeros_like(cov)
    for i in range(cov.shape[0]):
        for j in range(cov.shape[1]):
            corr[i, j] = cov[i, j] / (np.sqrt(cov[i, i]) * np.sqrt(cov[j, j]))            
    return corr
    

In [ ]:
show_plots = True
syst_dict = {}
syst_Flux_total_dict = {}
cov_type = "rate"

save_fig = True
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/Flux"
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)


    
for var_config in var_configs:
    total_cov_frac = None    
    total_cov = None    
    for syst_name in flux_systematics:
        univ_events, cv_events = get_univ_rates(cov_type = cov_type, evtdf = mc_evt_df, nudf = mc_nu_df, var_config = var_config, syst_name = syst_name, bkgd_subtract = True)
        ret_Flux = get_covariance_matrix(univ_events, cv_events)
        syst_dict[var_config.var_save_name + "_" + syst_name] = ret_Flux["cov_frac"]
        
        current_cov_frac = ret_Flux["cov_frac"]
        if total_cov_frac is None:
            # First iteration: initialize with the first matrix shape
            total_cov_frac = np.copy(current_cov_frac)
            total_cov = np.copy(ret_Flux["cov"])
        else:
            # Subsequent iterations: add the new matrix
            total_cov_frac += current_cov_frac
            total_cov += ret_Flux["cov"]
    syst_Flux_total_dict[var_config.var_save_name] = total_cov_frac
    
    if save_fig:
        total_corr = corr_from_cov(total_cov)

        f_name = f"flux_syst_{var_config.var_save_name}_frac_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov_frac, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"flux_syst_{var_config.var_save_name}_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"flux_syst_{var_config.var_save_name}_corr.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_corr, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                    save_fig=save_fig, save_name=save_full_path)
    
for var_config in var_configs:
    fig, ax = plt.subplots(figsize=(10,6))

    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])

    
    total_ret_frac = None
    
    # Store everything first
    syst_storage = []        
                
    # Store everything first
    syst_storage = []

    for syst_name in flux_systematics:
        syst = syst_dict[var_config.var_save_name + "_" + syst_name]
        syst_uncert = np.sqrt(np.diag(syst))
        
        if total_ret_frac is None:
            total_ret_frac = syst
        else:
            total_ret_frac += syst
    
        values = syst_uncert * 1e2
        integral = np.sum(values)

        syst_storage.append((integral, syst_name, values))

    # Sort by integral (largest first)
    syst_storage.sort(key=lambda x: x[0], reverse=True)

    # Keep only top 5
    top5 = syst_storage[:5]

    handles = []
    labels = []

    # ---- Plot Top 5 ----
    for integral, syst_name, values in top5:
        h = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=values,
            histtype="step",
            linewidth=2,
            label=flux_label_map[syst_name],
        )
        handles.append(h[2][0])
        labels.append(syst_name)

    # ---- Total ----
    frac_uncert_total = np.sqrt(np.diag(total_ret_frac))
    total_values = frac_uncert_total * 1e2

    total_handle = ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=total_values,
        histtype="step",
        linewidth=3,
        color="k",
        label="Total"
    )[2][0]

    handles.append(total_handle)
    labels.append("Total")

    # Legend
    ax.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        fontsize=11,
        frameon=True,
        edgecolor='gray'
    )

    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylim(0, max(total_values) * 1.4)
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")

    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()

    if save_fig:
        f_name = f"flux_syst_{var_config.var_save_name}_uncrt_breakdown.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        fig.savefig(save_full_path, format='pdf', bbox_inches='tight')
    

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/extended_flux_syst_dict.npz", **syst_Flux_total_dict)    

# G4

In [ ]:
g4_systematics = [
    'reinteractions_kminus_Geant4',
    'reinteractions_kplus_Geant4',
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4'
]

label_map = {
    "reinteractions_kminus_Geant4": r"$K^{-}$",
    "reinteractions_kplus_Geant4": r"$K^{+}$",
    "reinteractions_neutron_Geant4": "n",
    "reinteractions_piminus_Geant4": r"$\pi^{-}$",  # Changed # to \
    "reinteractions_piplus_Geant4": r"$\pi^{+}$",   # Changed # to \
    "reinteractions_proton_Geant4": "p",
}

In [ ]:
show_plots = True
syst_dict = {}
syst_g4_total_dict = {}
cov_type = "rate"
show_plots = True
#for cov_type in ["xsec", "rate"]:

save_fig = True
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/G4"
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)
    
for var_config in var_configs:
    total_cov_frac = None    
    total_cov = None    
    for syst_name in g4_systematics:
        univ_events, cv_events = get_univ_rates(cov_type = cov_type, evtdf = mc_evt_df, nudf = mc_nu_df, var_config = var_config, syst_name = syst_name, bkgd_subtract = True)
        
        ret_g4 = get_covariance_matrix(univ_events, cv_events)
        syst_dict[var_config.var_save_name + "_" + syst_name] = ret_g4["cov_frac"]
        
        current_cov_frac = ret_g4["cov_frac"]
        if total_cov_frac is None:
            # First iteration: initialize with the first matrix shape
            total_cov_frac = np.copy(current_cov_frac)
            total_cov = np.copy(ret_g4["cov"])
        
        else:
            # Subsequent iterations: add the new matrix
            total_cov_frac += current_cov_frac
            total_cov += ret_g4["cov"]
    syst_g4_total_dict[var_config.var_save_name] = total_cov_frac
    
    if save_fig:
        total_corr = corr_from_cov(total_cov)

        f_name = f"g4_syst_{var_config.var_save_name}_frac_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov_frac, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"g4_syst_{var_config.var_save_name}_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"g4_syst_{var_config.var_save_name}_corr.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_corr, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                    save_fig=save_fig, save_name=save_full_path)
        

# Assuming syst_dict, var_configs, and g4_systematics are already defined

for var_config in var_configs:
    # Use standard proportions to match the reference image
    fig, ax = plt.subplots(figsize=(8, 6))
    
    frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
    total_ret_frac = None
    # Store everything first
    syst_storage = []        
    # 1. Plot the dynamic systematics (G4, Genie, etc.)
    for syst_name in g4_systematics:
        # Retrieve the covariance matrix and calculate diagonal uncertainty
        syst = syst_dict[var_config.var_save_name + "_" + syst_name]
        syst_uncert = np.sqrt(np.diag(syst))
        
        if total_ret_frac is None:
            total_ret_frac = syst
        else:
            total_ret_frac += syst
        
        ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=label_map[syst_name],
            zorder=3
        )

    frac_uncert_total = np.sqrt(np.diag(total_ret_frac))
    total_values = frac_uncert_total * 1e2
    
    ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=total_values,
        histtype="step",
        linewidth=2,
        color="k",
        label="Total",
        zorder=5  # Ensure the total is always on top
    )

    # --- LEGEND & FORMATTING TO MATCH IMAGE ---
    
    # ncol=3 creates the multi-column look
    # loc="upper center" places it at the top of the axes
    ax.legend(
        loc="upper center", 
        ncol=3, 
        fontsize=11, 
        frameon=True,
        edgecolor='gray'
    )

    # Set limits and labels
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    # Multiplier 1.4 provides the 'white space' at the top for the legend
    ax.set_ylim(0, max(total_values) * 1.4) 
    
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")
    
    # Grid settings to match the crisp look of the plot
    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()


    if save_fig:
        f_name = f"g4_syst_{var_config.var_save_name}_uncrt_breakdown.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        fig.savefig(save_full_path, format='pdf', bbox_inches='tight')
        
    if show_plots:
        plt.show()


if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/extended_g4_syst_dict.npz", **syst_g4_total_dict)

# GENIE

In [ ]:
genie_label_map = {
    # --- CCQE / MEC ---
    "GENIEReWeight_SBN_v1_multisim_RPA_CCQE": "CCQE RPA Correction",
    "GENIEReWeight_SBN_v1_multisim_CoulombCCQE": "CCQE Coulomb Correction",
    "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape": "CCQE Form Factor Shape",
    "GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE": "CCQE z-exp Param 1",
    "GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE": "CCQE z-exp Param 2",
    "GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE": "CCQE z-exp Param 3",
    "GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE": "CCQE z-exp Param 4",
    "GENIEReWeight_SBN_v1_multisim_NormCCMEC": "CCMEC Normalization",
    "GENIEReWeight_SBN_v1_multisim_NormNCMEC": "NCMEC Normalization",
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC": "MEC Decay Angle",

    # --- Resonance (RES) ---
    "GENIEReWeight_SBN_v1_multisigma_MaCCRES": r"$M_A$ CCRES",
    "GENIEReWeight_SBN_v1_multisigma_MaNCRES": r"$M_A$ NCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvCCRES": r"$M_V$ CCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvNCRES": r"$M_V$ NCRES",
    "GENIEReWeight_SBN_v1_multisim_RDecBR1gamma": r"$X + \gamma$ Branching Ratio",
    "GENIEReWeight_SBN_v1_multisim_RDecBR1eta": r"$X + \eta$ Branching Ratio",
    "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi": r"$\Delta \to N\pi$ Ang. Dist.",
    "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad": r"$\Delta \to N\gamma$ Ang. Dist.",

    # --- Non-RES Background (p/n, CC/NC, 1pi/2pi) ---
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi": r"Non-RES $\nu p$ CC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi": r"Non-RES $\nu p$ CC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi": r"Non-RES $\nu p$ NC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi": r"Non-RES $\nu p$ NC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi": r"Non-RES $\nu n$ CC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi": r"Non-RES $\nu n$ CC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi": r"Non-RES $\nu n$ NC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi": r"Non-RES $\nu n$ NC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi": r"Non-RES $\bar{\nu} p$ CC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi": r"Non-RES $\bar{\nu} p$ CC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi": r"Non-RES $\bar{\nu} p$ NC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi": r"Non-RES $\bar{\nu} p$ NC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi": r"Non-RES $\bar{\nu} n$ CC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi": r"Non-RES $\bar{\nu} n$ CC $2\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi": r"Non-RES $\bar{\nu} n$ NC $1\pi$",
    "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi": r"Non-RES $\bar{\nu} n$ NC $2\pi$",

    # --- DIS / Coherent ---
    "GENIEReWeight_SBN_v1_multisigma_AhtBY": "Bodek-Yang DIS Aht",
    "GENIEReWeight_SBN_v1_multisigma_BhtBY": "Bodek-Yang DIS Bht",
    "GENIEReWeight_SBN_v1_multisigma_CV1uBY": "Bodek-Yang DIS CV1u",
    "GENIEReWeight_SBN_v1_multisigma_CV2uBY": "Bodek-Yang DIS CV2u",
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH": "CCCOH Normalization",
    "GENIEReWeight_SBN_v1_multisigma_NormNCCOH": "NCCOH Normalization",

    # --- FSI (Final State Interactions) ---
    "GENIEReWeight_SBN_v1_multisigma_MFP_pi": r"FSI $\pi$ Mean Free Path",
    "GENIEReWeight_SBN_v1_multisigma_FrCEx_pi": r"FSI $\pi$ Charge Exchange",
    "GENIEReWeight_SBN_v1_multisigma_FrInel_pi": r"FSI $\pi$ Inelastic",
    "GENIEReWeight_SBN_v1_multisigma_FrAbs_pi": r"FSI $\pi$ Absorption",
    "GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi": r"FSI $\pi$ Production",
    "GENIEReWeight_SBN_v1_multisigma_MFP_N": "FSI Nucleon MFP",
    "GENIEReWeight_SBN_v1_multisigma_FrCEx_N": "FSI Nucleon Charge Exchange",
    "GENIEReWeight_SBN_v1_multisigma_FrInel_N": "FSI Nucleon Inelastic",
    "GENIEReWeight_SBN_v1_multisigma_FrAbs_N": "FSI Nucleon Absorption",
    "GENIEReWeight_SBN_v1_multisigma_FrPiProd_N": "FSI Nucleon Pion Prod.",

    # --- Elastic ---
    "GENIEReWeight_SBN_v1_multisigma_MaNCEL": r"$M_A$ NC Elastic",
    "GENIEReWeight_SBN_v1_multisigma_EtaNCEL": r"$\eta$ NC Elastic",
}


genie_systematics_multisim = [
    'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
    'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
    'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
    'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
]

genie_systematics_multisigma = [
    "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
    "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
    "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
    "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
    'GENIEReWeight_SBN_v1_multisigma_AhtBY',
    'GENIEReWeight_SBN_v1_multisigma_BhtBY',
    'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
    'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
    "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
    'GENIEReWeight_SBN_v1_multisigma_MFP_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi',
    'GENIEReWeight_SBN_v1_multisigma_MFP_N',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
    'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
    'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
]

In [ ]:
import hashlib
for syst in genie_systematics_multisigma:
    print("Checking:", syst)

    morph_key = ('truth', syst, 'morph', '', '', '')
    ps_key    = ('truth', syst, 'ps1', '', '', '')

    if morph_key in mc_nu_df.columns:
        print("  Found morph")
        s_morph = mc_nu_df[morph_key]
        for i in range(100):
            '''
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            '''
            seed_input = str(i) + str(syst)
            seed_int = int(hashlib.md5(seed_input.encode()).hexdigest(), 16) % (2**32)
            np.random.seed(seed_int)
            wgt = (1 + (s_morph - 1) * 2 * np.abs(np.random.normal(0, 1))).clip(lower=0, upper=30)
            mc_nu_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt

    elif ps_key in mc_nu_df.columns:
        print("  Found ps1")
        s_ps = mc_nu_df[ps_key]
        for i in range(100):
            '''
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            '''
            seed_input = str(i) + str(syst)
            seed_int = int(hashlib.md5(seed_input.encode()).hexdigest(), 16) % (2**32)
            np.random.seed(seed_int)
            wgt = (1 + (s_ps - 1) * np.random.normal(0, 1)).clip(lower=0, upper=30)
            mc_nu_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt

In [ ]:
print(mc_nu_df.truth.GENIEReWeight_SBN_v1_multisigma_FrCEx_N.univ_0)

In [ ]:
missing_cols = mc_nu_df.columns.difference(mc_evt_df.columns)
cols_to_keep = list(missing_cols)
matchdf = ph.multicol_merge(
    mc_evt_df.reset_index(), 
    mc_nu_df[cols_to_keep].reset_index(), # Only merge the new stuff + keys
    left_on=[
        ("__ntuple", "","","","",""),
        ("entry", "","","","",""), 
        ("slc", "tmatch","idx","","","")
    ],
    right_on=[
        ("__ntuple", "","","","",""),
        ("entry", "","","","",""), 
        ("rec.mc.nu..index", "","","","","")
    ], 
    how="left"
)

matchdf = matchdf.set_index(mc_evt_df.index.names, verify_integrity=True)
mc_evt_df = matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])    

In [ ]:
show_plots = True
syst_dict = {}

syst_dict = {}
syst_genie_xsec_total_dict = {}
cov_type = "xsec"
show_plots = True

save_fig = True
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/genie"
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)
    
for var_config in var_configs:
    total_cov_frac = None    
    total_cov = None    
    
    for syst_name in genie_systematics_multisigma:
        univ_events, cv_events = get_univ_rates(cov_type=cov_type, evtdf=mc_evt_df, nudf=mc_nu_df, var_config=var_config, syst_name=syst_name, bkgd_subtract=True)
        ret_genie = get_covariance_matrix(univ_events, cv_events)
        syst_dict[var_config.var_save_name + "_" + syst_name] = ret_genie["cov_frac"]

        current_cov_frac = ret_genie["cov_frac"]
        if total_cov_frac is None:
            # First iteration: initialize with the first matrix shape
            total_cov_frac = np.copy(current_cov_frac)
            total_cov = np.copy(ret_genie["cov"])
        else:
            # Subsequent iterations: add the new matrix
            total_cov_frac += current_cov_frac
            total_cov += ret_genie["cov"]
            
    for syst_name in genie_systematics_multisim:
        univ_events, cv_events = get_univ_rates(cov_type=cov_type, evtdf=mc_evt_df, nudf=mc_nu_df, var_config=var_config, syst_name=syst_name, bkgd_subtract=True)
        ret_genie = get_covariance_matrix(univ_events, cv_events)
        syst_dict[var_config.var_save_name + "_" + syst_name] = ret_genie["cov_frac"]

        current_cov_frac = ret_genie["cov_frac"]
        if total_cov_frac is None:
            # First iteration: initialize with the first matrix shape
            total_cov_frac = np.copy(current_cov_frac)
            total_cov = np.copy(ret_genie["cov"])
        else:
            # Subsequent iterations: add the new matrix
            total_cov_frac += current_cov_frac
            total_cov += ret_genie["cov"]

            
    syst_genie_xsec_total_dict[var_config.var_save_name] = total_cov_frac
    if save_fig:
        total_corr = corr_from_cov(total_cov)

        f_name = f"genie_syst_{var_config.var_save_name}_frac_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov_frac, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"genie_syst_{var_config.var_save_name}_cov.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_cov, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                    save_fig=save_fig, save_name=save_full_path)
        
        f_name = f"genie_syst_{var_config.var_save_name}_corr.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        plot_heatmap(total_corr, 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                    save_fig=save_fig, save_name=save_full_path)
                    


for var_config in var_configs:
    fig, ax = plt.subplots(figsize=(9, 6))
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])

    total_ret_frac = None
    syst_storage = []

    for syst_name in genie_systematics_multisim:
        syst = syst_dict[var_config.var_save_name + "_" + syst_name]
        syst_uncert = np.sqrt(np.diag(syst))
        if total_ret_frac is None:
            total_ret_frac = syst
        else:
            total_ret_frac += syst
        values = syst_uncert * 1e2
        integral = np.sum(values)
        syst_storage.append((integral, syst_name, values))

    for syst_name in genie_systematics_multisigma:
        syst = syst_dict[var_config.var_save_name + "_" + syst_name]
        syst_uncert = np.sqrt(np.diag(syst))
        total_ret_frac += syst
        values = syst_uncert * 1e2
        integral = np.sum(values)
        syst_storage.append((integral, syst_name, values))

    # Sort by integral (largest first)
    syst_storage.sort(key=lambda x: x[0], reverse=True)

    # Keep only top 10
    top10 = syst_storage[:10]

    handles = []
    labels = []

    # Plot only top 5
    for integral, syst_name, values in top10:
        h = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=values,
            histtype="step",
            linewidth=2,
            label=genie_label_map[syst_name]
        )
        handles.append(h[2][0])
        labels.append(genie_label_map[syst_name]) # <--- CHANGE THIS from syst_name to the map lookup

    # ---- Total ----
    frac_uncert_total = np.sqrt(np.diag(total_ret_frac))
    total_values = frac_uncert_total * 1e2

    total_handle = ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=total_values,
        histtype="step",
        linewidth=3,
        color="k",
        label="Total"
    )[2][0]

    handles.append(total_handle)
    labels.append("Total")

    # Legend
    ax.legend(
        handles=handles,
        labels=labels,
        loc="upper center",
        ncol=2,
        fontsize=10,
        frameon=True,
        edgecolor='gray'
    )

    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylim(0, max(total_values) * 1.6)
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")

    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()

    if save_fig:
        f_name = f"genie_syst_{var_config.var_save_name}_uncrt_breakdown.pdf" 
        save_full_path = os.path.join(save_fig_dir, f_name)
        fig.savefig(save_full_path, format='pdf', bbox_inches='tight')


if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/extended_genie_xsec_syst_dict.npz", **syst_genie_xsec_total_dict)